# Ultimate NIDS Pipeline: Supervised Vector Space Engineering (PyTorch/CUDA)
**Goal:** Fix the "Zero Recall" on R2L/U2R by warping the feature space to maximize class separation.

**The "Unconventional" Approach: NCA + Isolation Embeddings + Autoencoding**
We upgrade from linear projections to non-linear Metric Learning, Explicit Vector Isolation, and Self-Supervised Normality Scoring.

**New Architecture:**
1.  **Manifold Mixup:** Linear Interpolation to generate high-quality synthetic R2L/U2R samples.
2.  **Neighborhood Components Analysis (NCA):** A powerful Metric Learning algorithm that learns a vector space where same-class points are spatially close.
3.  **Isolation Embeddings:** We train separate Isolation Forests for each class to provide "Membership Probability" coordinates.
4.  **Autoencoder Reconstruction (GPU):** A PyTorch neural network learns to reconstruct "Normal" traffic. The reconstruction error serves as a powerful "Weirdness Score".
5.  **Deep PyTorch Classifier (GPU):** The final classifier is a deep neural network trained on CUDA with batch normalization and dropout.

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
# Falls nicht installiert: pip install catboost
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("Warning: CatBoost not found. Falling back to Linear Head.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Processing on: {device}")

# ==========================================
# 1. SOTA PREPROCESSING (Feature Tokenization Prep)
# ==========================================
# Transformers brauchen "Tokens". Wir müssen Kategorien in Integers (0, 1, 2...)
# und Numerische Werte lassen wir, skalieren sie aber perfekt.

def load_and_prep():
    # Load Data (Pfade anpassen)
    df = pd.read_csv('Data/network_connections.csv')
    
    # Map Labels
    attack_map = {'normal': 'normal'}
    try:
        with open('Data/attack2category_map.txt', 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2: attack_map[parts[0]] = parts[1]
    except: pass
    
    df['label'] = df['label'].astype(str).str.replace('.', '', regex=False)
    df['category'] = df['label'].map(attack_map).fillna('other')
    
    # Categorical Features definieren
    cat_cols = ['protocol_type', 'service', 'flag', 'land', 'logged_in', 'is_host_login', 'is_guest_login']
    # Numerical Features (alles was übrig ist, außer Label)
    num_cols = [c for c in df.columns if c not in cat_cols + ['label', 'category']]
    
    # Feature Interaction (Domain Knowledge hilft dem Transformer)
    df['byte_ratio'] = np.log1p(df['src_bytes']) / (np.log1p(df['dst_bytes']) + 1)
    num_cols.append('byte_ratio')

    X = df.drop(['label', 'category'], axis=1)
    y = df['category']
    
    return X, y, cat_cols, num_cols

X, y, cat_cols, num_cols = load_and_prep()

# Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# --- TOKENIZATION PIPELINE ---
# 1. Categorical: Label Encoding für jede Spalte einzeln
cat_encoders = {}
X_train_cat = []
X_test_cat = []

for col in cat_cols:
    le = LabelEncoder()
    # Fit auf Train, unknown in Test auf 'other' mappen (dirty fix via string conversion)
    train_vals = X_train_raw[col].astype(str)
    test_vals = X_test_raw[col].astype(str)
    
    le.fit(train_vals)
    # Mapping für unbekannte Klassen im Testset
    classes = set(le.classes_)
    test_vals = test_vals.apply(lambda x: x if x in classes else list(classes)[0])
    
    X_train_cat.append(le.transform(train_vals))
    X_test_cat.append(le.transform(test_vals))
    cat_encoders[col] = len(classes) # Merken wie viele Kategorien pro Spalte

X_train_cat = np.stack(X_train_cat, axis=1).astype(np.int64)
X_test_cat = np.stack(X_test_cat, axis=1).astype(np.int64)

# 2. Numerical: Quantile Transform (Gauss)
qt = QuantileTransformer(output_distribution='normal', n_quantiles=2000, random_state=42)
X_train_num = qt.fit_transform(X_train_raw[num_cols]).astype(np.float32)
X_test_num = qt.transform(X_test_raw[num_cols]).astype(np.float32)

# 3. Target Encoding
le_y = LabelEncoder()
y_train_enc = le_y.fit_transform(y_train)
y_test_enc = le_y.transform(y_test)

print("Data Tokenized for Transformer.")

# ==========================================
# 2. THE FT-TRANSFORMER ARCHITECTURE
# ==========================================
class FTTransformer(nn.Module):
    def __init__(self, cat_cardinalities, num_numeric, d_token=64, n_layers=3, n_heads=8, d_ffn=128, dropout=0.1, n_classes=0):
        super().__init__()
        
        # 1. Feature Tokenizer
        # Für jede Kat-Spalte ein eigenes Embedding
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(card, d_token) for card in cat_cardinalities
        ])
        # Für jede Num-Spalte eine lineare Projektion in den d_token Raum + Bias
        self.num_embeddings = nn.ModuleList([
            nn.Linear(1, d_token) for _ in range(num_numeric)
        ])
        
        # [CLS] Token (wie bei BERT), aggregiert das Wissen
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token))
        
        # 2. Transformer Backbone
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, 
            nhead=n_heads, 
            dim_feedforward=d_ffn, 
            dropout=dropout, 
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # 3. Heads
        # Projection Head für Contrastive Learning (Vektorraum)
        self.proj_head = nn.Sequential(
            nn.Linear(d_token, d_token),
            nn.ReLU(),
            nn.Linear(d_token, 32) # Kleiner Raum für SupCon
        )
        
        # Classification Head (optional für End-to-End)
        self.cls_head = nn.Linear(d_token, n_classes)

    def forward(self, x_cat, x_num):
        batch_size = x_cat.shape[0]
        
        # Tokenization
        tokens = []
        
        # Categorical Embeddings
        for i, emb_layer in enumerate(self.cat_embeddings):
            tokens.append(emb_layer(x_cat[:, i]))
            
        # Numerical Embeddings (Input shape: Batch, 1)
        for i, lin_layer in enumerate(self.num_embeddings):
            tokens.append(lin_layer(x_num[:, i].unsqueeze(1)))
            
        # Stack tokens -> (Batch, Num_Features, d_token)
        x = torch.stack(tokens, dim=1)
        
        # Add [CLS] token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        
        # Transformer Pass
        x = self.transformer(x)
        
        # Wir nehmen nur den Output des [CLS] Tokens für die Vorhersage
        cls_output = x[:, 0, :]
        
        # Returns: Representation (64 dim), Projection (32 dim), Logits
        return cls_output, self.proj_head(cls_output), self.cls_head(cls_output)

# Config
model = FTTransformer(
    cat_cardinalities=list(cat_encoders.values()),
    num_numeric=X_train_num.shape[1],
    d_token=64,  # Embedding Size
    n_layers=3,  # Tiefe
    n_heads=4,   # Attention Heads
    n_classes=len(le_y.classes_)
).to(device)

# ==========================================
# 3. SUPERVISED CONTRASTIVE LOSS (SOTA)
# ==========================================

# Zieht alle Samples der GLEICHEN Klasse zusammen, egal wie unterschiedlich sie aussehen.
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: (Batch, Dim) - L2 normalized
        # labels: (Batch)
        features = F.normalize(features, dim=1)
        
        # Similarity Matrix
        sim_matrix = torch.matmul(features, features.T) / self.temperature
        
        # Maske für "Self" (Diagonale entfernen)
        mask = torch.eye(labels.shape[0], device=labels.device).bool()
        
        # Maske für Positive (Gleiche Klasse)
        labels = labels.view(-1, 1)
        mask_pos = torch.eq(labels, labels.T).bool()
        mask_pos = mask_pos ^ mask # Entferne Self-Match
        
        # Wenn keine Positives im Batch, return 0 (vermeidet NaN)
        if not mask_pos.any(): return torch.tensor(0.0, device=device, requires_grad=True)

        # LogSumExp Trick für den Nenner
        exp_sim = torch.exp(sim_matrix) * (~mask).float() # Self matches raus
        log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-6)
        
        # Mean log-likelihood über alle Positives
        mean_log_prob_pos = (mask_pos * log_prob).sum(dim=1) / (mask_pos.sum(dim=1) + 1e-6)
        
        loss = -mean_log_prob_pos
        return loss.mean()

# ==========================================
# 4. TRAINING: PHASE 1 (VECTOR SPACE LEARNING)
# ==========================================
# Weighted Sampler für Imbalance
class_counts = np.bincount(y_train_enc)
weights = 1. / class_counts
samples_weights = weights[y_train_enc]
sampler = WeightedRandomSampler(torch.from_numpy(samples_weights), len(samples_weights))

# DataLoader
train_ds = TensorDataset(
    torch.tensor(X_train_cat).to(device), 
    torch.tensor(X_train_num).to(device), 
    torch.tensor(y_train_enc).to(device)
)
train_dl = DataLoader(train_ds, batch_size=256, sampler=sampler) # Shuffle im Sampler integriert

optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
criterion_sup = SupConLoss(temperature=0.1)
criterion_ce = nn.CrossEntropyLoss()

print("\n--- Phase 1: Context-Aware Vector Space Learning (SupCon) ---")
# Der Transformer lernt hier NUR, wie Daten zusammengehören (Clustering), keine Klassifizierung!

for epoch in range(15):
    model.train()
    loss_acc = 0
    for cat, num, y in train_dl:
        optimizer.zero_grad()
        # Projection Head nutzen für SupCon
        _, proj, logits = model(cat, num)
        
        # SupCon Loss (Lernt Struktur) + bisschen CE Loss (Lernt Klassen)
        loss = criterion_sup(proj, y) + 0.5 * criterion_ce(logits, y)
        
        loss.backward()
        optimizer.step()
        loss_acc += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {loss_acc/len(train_dl):.4f}")

# ==========================================
# 5. TRAINING: PHASE 2 (HYBRID CATBOOST)
# ==========================================
print("\n--- Phase 2: Extracting Super Vectors & Training CatBoost ---")
# Wir nutzen jetzt den Transformer NUR als Feature Extractor.
# Er hat verstanden, wie "Source Bytes" mit "Protocol" interagiert.
# Diese "Context Vectors" geben wir an CatBoost.

def get_vectors(loader):
    model.eval()
    vectors = []
    labels = []
    with torch.no_grad():
        for cat, num, y in loader:
            rep, _, _ = model(cat, num) # Repräsentation vom [CLS] Token
            vectors.append(rep.cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(vectors), np.concatenate(labels)

# Ganzen Datensatz durch den Transformer jagen (ohne Sampler, sequential)
train_dl_seq = DataLoader(train_ds, batch_size=512, shuffle=False)
test_ds = TensorDataset(
    torch.tensor(X_test_cat).to(device), 
    torch.tensor(X_test_num).to(device), 
    torch.tensor(y_test_enc).to(device)
)
test_dl = DataLoader(test_ds, batch_size=512, shuffle=False)

X_train_trans, y_train_trans = get_vectors(train_dl_seq)
X_test_trans, y_test_trans = get_vectors(test_dl)

print(f"Transformer Vectors Created. Shape: {X_train_trans.shape}")

# Train Gradient Boosting on Transformer Vectors
# CatBoost ist extrem robust und holt die letzten % raus
if HAS_CATBOOST:
    print("Training CatBoost on Deep Embeddings...")
    # Class Weights für CatBoost berechnen
    classes = np.unique(y_train_trans)
    weights = len(y_train_trans) / (len(classes) * np.bincount(y_train_trans))
    class_weights_map = {i: w for i, w in enumerate(weights)}
    
    clf = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function='MultiClass',
        class_weights=class_weights_map,
        verbose=100,
        task_type="GPU" if device.type == 'cuda' else "CPU"
    )
    clf.fit(X_train_trans, y_train_trans, eval_set=(X_test_trans, y_test_trans), early_stopping_rounds=50)
    
    print("\n--- Final Evaluation (Hybrid SOTA) ---")
    y_pred = clf.predict(X_test_trans)
    print(classification_report(y_test_trans, y_pred, target_names=le_y.classes_))
else:
    print("Training Linear Head (Fallback)...")
    # Falls kein CatBoost, einfaches LogReg auf den Vektoren
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(class_weight='balanced', max_iter=1000)
    clf.fit(X_train_trans, y_train_trans)
    y_pred = clf.predict(X_test_trans)
    print(classification_report(y_test_trans, y_pred, target_names=le_y.classes_))

🚀 Processing on: cuda
Data Tokenized for Transformer.

--- Phase 1: Context-Aware Vector Space Learning (SupCon) ---
Epoch 1 Loss: 4.3151
Epoch 2 Loss: 4.0367
Epoch 3 Loss: 4.0055
Epoch 4 Loss: 3.9951
Epoch 5 Loss: 3.9828
Epoch 6 Loss: 3.9742
Epoch 7 Loss: 3.9711
Epoch 8 Loss: 3.9688
Epoch 9 Loss: 3.9653
Epoch 10 Loss: 3.9621


KeyboardInterrupt: 